In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image

from torchvision import models
from torchvision import transforms

from ultralytics import YOLO

from scipy.stats import entropy

from tqdm import tqdm

### Configuration & Paths

In [2]:
# ==========================
# CONFIGURATION
# ==========================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEST_PATH = Path("data/test")

MODEL_PATH = "best_model.pth"
XGB_MODEL_PATH = "best_xgboost_model.pkl"

print(f"Device : {DEVICE}")
print(f"Test Images : {len(list(TEST_PATH.glob('*')))}")

Device : cuda
Test Images : 300


In [3]:
# ==========================
# IMAGE TRANSFORM
# ==========================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform Loaded Successfully!")

Transform Loaded Successfully!


### Load ResNet18 & Create Feature Extractor

In [4]:
# ==========================
# LOAD RESNET18
# ==========================

model = models.resnet18(weights=None)

# Replace FC Head (same as training)
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.fc.in_features, 3)
)

# Load trained weights
model.load_state_dict(
    torch.load(MODEL_PATH, map_location=DEVICE)
)

model = model.to(DEVICE)
model.eval()

print("ResNet Model Loaded Successfully!")

ResNet Model Loaded Successfully!


In [5]:
# ==========================
# FEATURE EXTRACTOR
# ==========================

feature_extractor = nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

print("ResNet Feature Extractor Ready!")

ResNet Feature Extractor Ready!


In [6]:
print(feature_extractor)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (

### Load YOLOv8 & Register Feature Hook

In [7]:
# ==========================
# LOAD YOLO MODEL
# ==========================

yolo_model = YOLO("yolov8n.pt")

print("YOLO Model Loaded Successfully!")

YOLO Model Loaded Successfully!


### Register the Layer 21 Hook

In [8]:
# ==========================
# YOLO FEATURE HOOK
# ==========================

feature_maps = {}

def hook_fn(module, input, output):
    feature_maps["layer21"] = output

hook = yolo_model.model.model[21].register_forward_hook(hook_fn)

print("YOLO Layer 21 Hook Registered!")

YOLO Layer 21 Hook Registered!


In [9]:
print(yolo_model.model.model[21])

C2f(
  (cv1): Conv(
    (conv): Conv2d(384, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (cv2): Conv(
    (conv): Conv2d(384, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (m): ModuleList(
    (0): Bottleneck(
      (cv1): Conv(
        (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act)